# A demonstrative code to show the simulations performed & analytical functions plotted found in the first results chapter of my thesis.

This code is split into constant growth simulations and exponentially decaying growth simulations.


In [ ]:
import pandas as pd
import numpy as np
import scipy.integrate
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from numba import njit, jit, cfunc
import pickle
import sys
import importlib
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import matplotlib.cm as mplcm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.optimize import curve_fit
from matplotlib.colors import LinearSegmentedColormap
from dataclasses import dataclass
from matplotlib.colors import LogNorm
import itertools
import math
from scipy.signal import savgol_filter

sys.path.append('/Users/bowena/Documents/GitHub/GrowthSolverTest/')

import FuncPatternQuant
import FuncAnalytical
import FuncSolver

In [22]:
read_variables = np.loadtxt("Variables.txt") 
read_parameters = np.loadtxt("Parameters.txt") 

[D,sMag,L0,sRatio,epsilon,dt_factor,F,no_stores,N,m_val,sigma] = read_parameters


alpha_values = read_variables[:,0]
g_max_values = read_variables[:,1]
k_values = read_variables[:,2]
G_values = np.round(g_max_values/(alpha_values*(1+epsilon)),14)

k_set = np.sort(list(set(k_values)))
G_set = np.sort(list(set(G_values)))
alpha_set = np.sort(list(set(alpha_values)))

k = k_set[1]
g_max = G_set[3]


# Constant growth 


# $g = g_0$

In [ ]:
N = int(N)
N = 201
Y = np.linspace(0,L0,N) # co-moving coordinates (x coordinates at t=0)
c_init = np.zeros(N) # initial concentration
dy = L0/(N-1) # space step
w = round(L0*sRatio/dy)+1 # the index for the edge of the source: so the source spans from 0 to sSize0 - round this down 
N_1 = N-1
N_2 = N-2

dt = dt_factor*(2/(4*D/(dy**2) +k+g_max))   # setting dt according to k and g_max, via the vN stability criterion
t_m = 20/k # estimate of morphogen time to reach SS 
fold_change = 5 
gamma = 1

t_g = FuncAnalytical.find_t_g_constant_growth(L0,g_max,epsilon,fold_change*L0)
t_max =  t_g
count_max = round(t_max/dt) # the number of time points that will be computed, rounded up
store_no = math.ceil(count_max/(no_stores)) # the number of counts at which I store the results.

In [ ]:
timeArray, xArray, MArray, x_YArray, LArray, wArray = FuncSolver.M_solver_ConstantGrowth_scale_source(N,N_1,N_2,w,D,k,epsilon,sRatio,gamma,sMag,dt,dy,L0,store_no,c_init,count_max,g_max)

In [ ]:
prefactorArray, decay_lengthArray, perrArray = FuncPatternQuant.find_decay_length_array(xArray,MArray,wArray,N,sMag,D,k,L0,sRatio)

RArray = decay_lengthArray/LArray

In [ ]:
t_p, t_n, max_tau  = FuncPatternQuant.find_max_t_sigma(RArray,timeArray,m_val,sigma)


# Exponentially decaying growth

# $ g = g_0 e^{-\alpha t}$

In [ ]:
timeArray, xArray, cArray, gArray, x_YArray, LArray, wArray = FuncSolver.M_solver_Growth_scale_source(N,N_1,N_2,w,D,k,epsilon,sRatio,gamma,sMag,dt,dy,L0,store_no,c_init,count_max,g_max,alpha)
